# A second SAE dictionary on both contrast arms: the gap replicates **at the layer it is quoted at**

**Job `50604600`** (`scripts/slurm/sudoku2_sae_kindsplit_seed1.sbatch`), submitted 2026-10-31 22:25:03,
started 22:34:14, ended 23:31:20, elapsed **57:06**, exit 0:0, node `sh04-03n01`.
**Repo at `d0b4654`; `spin_gnn/sudoku_sae.py` at `10c38b0`; seed 1.**

Finding 37 (`50536365`) re-fit the **solver** at a second seed. This job re-fits the other two arms --
the **0.854 leak-free TWIN** and the **0.9960 LEAK-READER** -- which are the pair findings 19 and 21's
headline contrast is actually measured on. The contrast therefore has an error bar on *both* sides for
the first time.

## Headline

At layer 8, the layer the project quotes, the twin-minus-leak-reader gap in `best_justifier_f1`
at `pre_color` moves **+0.496 -> +0.487**, a shift of 0.009. **The contrast replicates.**

But the pre-registration in the job header asked for the *max over layers* for both arms, and by that
statistic the gap is **+0.233 at seed 0 and +0.202 at seed 1 -- both below the pre-registered 0.30 bar.**
That is an arithmetic error in my own pre-registration, not a replication failure: the 0.496 baseline
I wrote into the header is the twin's **max** against the leak-reader's value **at L8**. Pre-registration
(a) is scored FAILED below, at both seeds, and the lesson is recorded rather than quietly rescored.

## What the two scorings actually mean

The two models differ in **when** the forcing set appears, not in **whether** it does. The twin has
`justifies_next` features by L8, inside finding 27's L6->L10 activation-patching handoff window; the
leak-reader has none until L10, after that window closes. Finding 29(e) noted this in passing -- this
run makes it the headline, and every statement of the between-arm gap now has to name its layer.

## Comparison to the previous best on the same eval

The previous best on this eval is the seed-0 pair of dictionaries themselves (`49744985`, 10-27,
findings 19/21) -- the same checkpoints, same SAE configuration, same 256-predicate scoring. This
notebook re-reads those CSVs and puts the two seeds side by side; there is no earlier two-seed
measurement on these arms to compare against.

In [ ]:
import json, os
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# `display` is a Jupyter builtin; fall back so the same cells run headless.
try:
    display
except NameError:
    def display(x): print(x.to_string() if hasattr(x, "to_string") else x)

S = os.environ["SCRATCH"] + "/spin_gnn"
RUNS = {("TWIN", 0): f"{S}/sudoku_sae_kindsplit_2026-10-27/leakfree_l0_64",        # 49744985
        ("LEAK", 0): f"{S}/sudoku_sae_kindsplit_2026-10-27/leaked_l0_64",          # 49744985
        ("TWIN", 1): f"{S}/sudoku_sae_kindsplit_2026-10-31_seed1/leakfree_l0_64_seed1",
        ("LEAK", 1): f"{S}/sudoku_sae_kindsplit_2026-10-31_seed1/leaked_l0_64_seed1"}
# The solver's two kind-split dictionaries (findings 19 and 37): not an arm of this
# contrast, but they carry the same pre_node measurement, so the asymmetry tally
# below is counted over all six dictionaries rather than just these four.
SOLVER = {("SOLVER", 0): f"{S}/sudoku_sae_solved_2026-10-28/leakfree1000_l0_64",
          ("SOLVER", 1): f"{S}/sudoku_sae_solved_2026-10-31/leakfree1000_l0_64_seed1"}
QUOTED = 8     # the layer findings 19/21 quote the contrast at
BAR = 0.30     # the bar written into the job header

def _fam(t):
    t = t.copy()
    t["jn"] = t.family_counts.map(lambda s: json.loads(s).get("justifies_next", 0))
    t["nn"] = t.n_next_node_features
    return t

PC, PN, FIT, PN_ALL = {}, {}, {}, {}
for key, p in {**RUNS, **SOLVER}.items():
    k = _fam(pd.read_csv(f"{p}/sae_summary_by_kind.csv"))
    PN_ALL[key] = k[k.position_kind == "pre_node"].set_index("layer").sort_index()
    if key not in RUNS:
        continue
    PC[key] = k[k.position_kind == "pre_color"].set_index("layer").sort_index()
    PN[key] = PN_ALL[key]
    FIT[key] = pd.read_csv(f"{p}/sae_summary.csv").set_index("layer").sort_index()

LAYERS = list(PC[("TWIN", 1)].index)
print("layers:", LAYERS)
print("dictionaries:", {f"{a} seed {s}": len(PC[(a, s)]) for a, s in RUNS})

## 1. Metrics table -- `best_justifier_f1` and the `justifies_next` feature count, at `pre_color`

`best_justifier_f1` is the best F1 over the 256 scored predicates for `justifies_next[u]` ("u is a KNOWN
neighbour of the node about to be emitted"); the count is how many dictionary features pass the
interpretability screen for that family. The count is the noisier of the two and was never pre-registered.

In [ ]:
rows = []
for L in LAYERS:
    r = {"layer": L}
    for a in ("TWIN", "LEAK"):
        for s in (0, 1):
            r[f"{a[0].lower()}{s}_f1"] = PC[(a, s)].best_justifier_f1[L]
            r[f"{a[0].lower()}{s}_n"] = int(PC[(a, s)].jn[L])
    r["gap_s0"] = r["t0_f1"] - r["l0_f1"]
    r["gap_s1"] = r["t1_f1"] - r["l1_f1"]
    rows.append(r)
tbl = pd.DataFrame(rows).set_index("layer")
display(tbl.round(4))

tot = {f"{a} seed {s}": int(PC[(a, s)].jn.sum()) for a in ("TWIN", "LEAK") for s in (0, 1)}
print("\ntotal justifies_next features at pre_color:", tot)

### The headline, scored the two ways the pre-registration conflated

In [ ]:
g = {}
for s in (0, 1):
    t, l = PC[("TWIN", s)], PC[("LEAK", s)]
    g[s] = dict(at=t.best_justifier_f1[QUOTED] - l.best_justifier_f1[QUOTED],
                mx=t.best_justifier_f1.max() - l.best_justifier_f1.max(),
                t_at=t.best_justifier_f1[QUOTED], l_at=l.best_justifier_f1[QUOTED],
                t_max=t.best_justifier_f1.max(), l_max=l.best_justifier_f1.max(),
                t_arg=int(t.best_justifier_f1.idxmax()), l_arg=int(l.best_justifier_f1.idxmax()))

print(f"AT L{QUOTED} (the layer the project quotes)")
for s in (0, 1):
    print(f"  seed {s}: twin {g[s]['t_at']:.4f} - leak {g[s]['l_at']:.4f} = {g[s]['at']:+.4f}")
print(f"  movement between seeds: {abs(g[1]['at'] - g[0]['at']):.4f}   -> REPLICATES")

print(f"\nMAX OVER LAYERS (what the header asked for)")
for s in (0, 1):
    print(f"  seed {s}: twin {g[s]['t_max']:.4f} (L{g[s]['t_arg']}) - "
          f"leak {g[s]['l_max']:.4f} (L{g[s]['l_arg']}) = {g[s]['mx']:+.4f}"
          f"   {'PASSES' if g[s]['mx'] >= BAR else 'FAILS'} the {BAR} bar")
print("\nThe 0.496 baseline written into the job header is g[0]['t_max'] - g[0]['l_at'] = "
      f"{g[0]['t_max'] - g[0]['l_at']:.4f}: a max on one side, a layer value on the other.")

## 2. The `pre_node` side -- the asymmetry findings 21 and 29 rest on

`pre_node` is the position just before the model emits a **node id**; `pre_color` is just before it emits
that node's **colour**. If the forcing set were a general property of the residual stream it would be
readable at both. It is readable at neither `pre_node` dictionary, at any layer, in either arm.

In [ ]:
pn = pd.DataFrame({f"{a} s{s} n": [int(PN[(a, s)].jn[L]) for L in LAYERS]
                   for a in ("TWIN", "LEAK") for s in (0, 1)}, index=LAYERS)
for a in ("TWIN", "LEAK"):
    for s in (0, 1):
        pn[f"{a} s{s} F1"] = [PN[(a, s)].best_justifier_f1[L] for L in LAYERS]
        pn[f"{a} s{s} null"] = [PN[(a, s)].median_best_f1_null[L] for L in LAYERS]
pn.index.name = "layer"
display(pn.round(4))

cells = sum(len(t) for t in PN_ALL.values())
print(f"\njustifies_next features at pre_node, summed over every kind-split dictionary ever fit in "
      f"this project -- {len(PN_ALL)} dictionaries, {cells} layer-measurements, three checkpoints "
      f"x two seeds: {int(sum(t.jn.sum() for t in PN_ALL.values()))}")
for k, t in PN_ALL.items():
    print(f"   {k[0]} seed {k[1]}: {int(t.jn.sum())} features, max best F1 "
          f"{t.best_justifier_f1.max():.4f} vs null {t.median_best_f1_null.max():.4f}")

## 3. Were the two seeds fit under the same pressure?

A smaller gap would mean nothing if seed 1's dictionaries were simply looser fits. They are not:
`l1_final` is identical across seeds at every layer of both arms **from L2 up**, variance explained
agrees to four decimals, and achieved L0 is within about 1 in 60 of the target of 64. The one exception
is the twin's layer 0, where the l1 search bottoms out near zero and the two seeds differ by 2e-05 on
a coefficient of 2e-05 -- both print as 0.0000 in the table below. **Layer 0 is degenerate in all four fits** --
L0 collapses to 6-8 with ~3932 of 4096 features dead -- exactly as it was in the solver (finding 37).
That is a property of the layer, not of a seed.

In [ ]:
fit = pd.DataFrame(index=LAYERS)
fit.index.name = "layer"
for a in ("TWIN", "LEAK"):
    for s in (0, 1):
        fit[f"l1 {a} s{s}"] = [FIT[(a, s)].l1_final[L] for L in LAYERS]
for a in ("TWIN", "LEAK"):
    for s in (0, 1):
        fit[f"varexp {a} s{s}"] = [FIT[(a, s)].frac_variance_explained[L] for L in LAYERS]
for a in ("TWIN", "LEAK"):
    for s in (0, 1):
        fit[f"L0 {a} s{s}"] = [FIT[(a, s)].l0[L] for L in LAYERS]
display(fit.round(4))

for a in ("TWIN", "LEAK"):
    d = (fit[f"l1 {a} s0"] - fit[f"l1 {a} s1"]).abs()
    print(f"\nl1_final {a}: exactly equal from L2 up: {bool((d[d.index >= 2] == 0).all())}; "
          f"layer-0 values {fit[f'l1 {a} s0'][0]:.2e} vs {fit[f'l1 {a} s1'][0]:.2e}")
for a in ("TWIN", "LEAK"):
    for s in (0, 1):
        d = FIT[(a, s)].n_dead_features[0]
        print(f"  layer 0, {a} seed {s}: L0 {FIT[(a, s)].l0[0]:.1f}, {int(d)} dead of "
              f"{int(FIT[(a, s)].d_sae[0])}")

### Configuration, identical across every cell

Expansion 16 (`d_sae` = 4096 on `d_model` 256), 1500 SAE epochs, 3000 instances, 40 positions each,
alive-rate threshold 1e-3, 19 null shifts, target L0 64, `--split-by-kind`, layers 0/2/4/6/8/10/12.
`--seed 1` moves four things together in `sudoku_sae`: the instance pool, which positions are sampled,
the SAE initialisation and the null-shift RNG. So this is a full replication, not an init-only ablation --
agreement is strong evidence, and a disagreement could not have been pinned on any one of the four.

**Code delta.** `spin_gnn/sudoku_sae.py` changed exactly once between the seed-0 arms (10-27) and this
job: commit `10c38b0` (10-28) **added two writes** -- the per-dimension normalisation statistics the
ablation work needs, and the per-kind per-feature CSVs. Nothing in the fitting or scoring path moved,
which the identical `l1_final` values above confirm independently of reading the diff.

## 4. Key plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
STYLE = {("TWIN", 0): ("#2f855a", "-", "o"), ("TWIN", 1): ("#68a68a", "--", "s"),
         ("LEAK", 0): ("#9b2c2c", "-", "o"), ("LEAK", 1): ("#c08080", "--", "s")}
NAME = {"TWIN": "0.854 TWIN (leak-free)", "LEAK": "LEAK-READER (0.9960)"}

for ax, (D, title) in zip(axes, [(PC, "pre_color: picking the COLOUR"),
                                 (PN, "pre_node: picking the NODE  (dotted = own null)")]):
    for key, (c, ls, m) in STYLE.items():
        t = D[key]
        ax.plot(t.index, t.best_justifier_f1, ls, color=c, marker=m,
                label=f"{NAME[key[0]]}, seed {key[1]}")
        if D is PN:
            ax.plot(t.index, t.median_best_f1_null, ":", color="0.6", lw=1)
    ax.set_xlabel("layer"); ax.set_ylim(-0.03, 1.05); ax.set_title(title)
axes[0].axvspan(6, 10, color="#f6e05e", alpha=0.25)
axes[0].text(6.2, 0.08, "L6-L10: activation-patching\nhandoff window (finding 27)",
             color="#975a16", fontsize=9)
axes[0].set_ylabel("best F1 for any justifies_next predicate")
axes[0].legend(fontsize=8, loc="upper left")
fig.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for s, (c, ls, m) in [(0, ("#2b6cb0", "-", "o")), (1, ("#90b8dd", "--", "s"))]:
    gap = [PC[("TWIN", s)].best_justifier_f1[L] - PC[("LEAK", s)].best_justifier_f1[L] for L in LAYERS]
    ax.plot(LAYERS, gap, ls, color=c, marker=m, label=f"seed {s}")
ax.axvspan(6, 10, color="#f6e05e", alpha=0.25)
ax.axhline(0, color="0.6", lw=1)
ax.set_xlabel("layer"); ax.set_ylabel("twin minus leak-reader, best justifies_next F1")
ax.set_title("The gap is LAYER-LOCAL"); ax.legend(fontsize=9)

ax = axes[1]
vals = [g[0]["at"], g[1]["at"], g[0]["mx"], g[1]["mx"]]
cols = ["#2f855a", "#68a68a", "#9b2c2c", "#c08080"]
labs = [f"seed 0\nat L{QUOTED}", f"seed 1\nat L{QUOTED}", "seed 0\nmax over L", "seed 1\nmax over L"]
ax.bar(range(4), vals, color=cols)
for i, v in enumerate(vals):
    ax.text(i, v + 0.012, f"{v:+.3f}", ha="center", fontsize=10)
ax.axhline(BAR, ls="--", color="0.35")
ax.text(3.45, BAR + 0.012, f"pre-registered bar {BAR}", ha="right", fontsize=9, color="0.35")
ax.set_xticks(range(4)); ax.set_xticklabels(labs, fontsize=9)
ax.set_ylim(0, 0.58); ax.set_title("The same headline, scored two ways")
fig.tight_layout(); plt.show()

## 5. The four pre-registrations, scored

Written into `scripts/slurm/sudoku2_sae_kindsplit_seed1.sbatch` before submission, against seed 0's
exact table.

**(a) HEADLINE** -- max-over-layers twin-minus-leak-reader gap at `pre_color` is at least 0.30.
**FAILED AS WRITTEN, AND IT WOULD HAVE FAILED AT SEED 0 TOO:** +0.202 at seed 1, +0.233 at seed 0.
The 0.496 baseline in the header is the twin's max against the leak-reader's L8 value. Scored at L8,
the layer the project actually quotes, it is +0.496 -> +0.487 and replicates.

**(b) ASYMMETRY** -- zero `justifies_next` features at `pre_node`, all seven layers, both arms.
**CONFIRMED** -- 0 at every layer in both arms, best F1 0.229 and 0.253 against nulls of 0.102 and 0.098.
That makes **six kind-split dictionaries and 42 layer-measurements at zero**, across three checkpoints
and two seeds.

**(c) LAYER 0** -- `best_next_node_f1` >= 0.99 at `pre_color` and zero `next_node` features at `pre_node`,
in both arms (counts at `pre_color` deliberately NOT pre-registered). **CONFIRMED** -- F1 1.0000 and
1.0000 at `pre_color`, 0 features at `pre_node` in both. Not pre-registering the counts was right:
they went 23->50 and 24->50 while F1 held at 1.000, the same instability the solver showed.

**(d) LOW CONFIDENCE** -- the leak-reader's `justifies_next` features stay LATE, at L10/L12, with none at
L6 or L8. **CONFIRMED, and tighter than predicted:** 13 at L10 and 0 at L12 (seed 0 had 13 and 2), zero
at L6 and L8 in both seeds. The falsifier was any leaked feature at L6 or L8; it did not happen.

3 of 4, with the miss on the one that mattered and for a reason that has nothing to do with the data.
The lesson is narrow and worth keeping: **write the pre-registered statistic as the expression you will
compute, not as a sentence plus a remembered number.**

In [ ]:
print(f"(b) pre_node justifies_next features over all {len(PN_ALL)} kind-split dictionaries:",
      int(sum(t.jn.sum() for t in PN_ALL.values())))
print()
for a in ("TWIN", "LEAK"):
    for s in (0, 1):
        print(f"(c) {a} s{s}: layer-0 best_next_node_f1 at pre_color "
              f"{PC[(a, s)].best_next_node_f1[0]:.4f}, n_next_node_features "
              f"pre_color {int(PC[(a, s)].nn[0])} / pre_node {int(PN[(a, s)].nn[0])}")
print()
for s in (0, 1):
    print(f"(d) LEAK s{s} justifies_next counts by layer: "
          f"{dict(zip(LAYERS, PC[('LEAK', s)].jn.astype(int)))}")

## 6. What this buys, and what it does not

**Buys.** The contrast findings 19 and 21 are built on is no longer one dictionary per arm. At L8 it
moves 0.009 between seeds against a gap of 0.487, and the `pre_node` asymmetry is now zero-for-zero in
six dictionaries -- 42 layer-measurements across three checkpoints and two seeds.

**Has to be reworded.** Every place the project states the between-arm gap, it must name the layer.
*"The leak-free model represents the forcing set and the leaked one does not"* is false as a global
claim -- the leak-reader has 13 forcing-set features at L10 at F1 0.519. The true claim is about
**timing**: the twin has the forcing set inside the causal handoff window, the leak-reader only after it.

**Still unfixed.** `best_f1` is a max over 256 predicates in every seed, so the multiple-comparisons
ceiling is unchanged -- two seeds showing the same ceiling gives the same answer twice, which is weaker
than removing it. Two seeds is two, not a distribution. And the SAE stays **correlational**: the
feature-level ablation (finding 30) found no sensitivity at all, so what the model *uses* comes from the
interventions in findings 27 and 28, not from here.

**What would settle the timing claim.** Activation patching on the LEAK-READER, the same L6-L10 sweep
finding 27 ran on the two leak-free models. If the timing story is right, its forcing fact should arrive
late or not at all -- though it already flips at only 0.388, so there may be little to track.

**Outputs** under `$SCRATCH/spin_gnn/sudoku_sae_kindsplit_2026-10-31_seed1/` (90-day purge; the summaries
are small and the dictionaries are re-derivable from the checkpoints).
**Slides:** `notebooks/sudoku_sae_arms_seed1_2026-10-31/slides.pdf`.